# shap-relativities

**Multiplicative rating relativities from GBMs, in the same format as GLM exp(beta) factors.**

This notebook shows the core workflow in under two minutes: train a Poisson CatBoost model on synthetic UK motor data, extract SHAP-based relativities per rating factor level, validate the reconstruction, and compare against the known true parameters.

The problem this solves: your GBM sits in a notebook outperforming the GLM in production, but you cannot get a factor table out of it. The regulator wants a relativity table. `shap-relativities` closes that gap.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/shap-relativities/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q "shap-relativities[ml]" catboost polars

## Synthetic data

We build 3,000 synthetic motor policies with three rating factors:

- `area_code` - six area bands (0-5), coarser version of A-F
- `ncd_years` - no-claims discount years (0-5), true discount is `exp(-0.12 * ncd)`
- `has_convictions` - binary (0/1), true loading is `exp(0.45) = 1.57`

The true coefficients are known, so we can measure how accurately SHAP recovers them.

In [ ]:
import numpy as np
import polars as pl
import catboost

rng = np.random.default_rng(42)
n = 3_000

area_code       = rng.integers(0, 6, n)
ncd_years       = rng.integers(0, 6, n)
has_convictions = rng.integers(0, 2, n)
exposure        = rng.uniform(0.5, 1.0, n)

# Area relativities: band 0 = base (1.0), band 5 = 1.6x
area_coeff = np.array([0.0, 0.10, 0.14, 0.24, 0.46, 0.65])
log_mu = (
    -2.8
    + area_coeff[area_code]
    - 0.12 * ncd_years
    + 0.45 * has_convictions
    + np.log(exposure)
)
mu = np.exp(log_mu)
claim_count = rng.poisson(mu).astype(float)

df = pl.DataFrame({
    "area_code":       area_code.astype(np.int32),
    "ncd_years":       ncd_years.astype(np.int32),
    "has_convictions": has_convictions.astype(np.int32),
    "claim_count":     claim_count,
    "exposure":        exposure,
})

print(f"Policies: {n:,}  |  Claim count: {int(claim_count.sum())}  |  Loss ratio: {claim_count.sum()/mu.sum():.3f}")
df.head()

## Train a Poisson CatBoost model

CatBoost with `loss_function="Poisson"` fits a log-link model, which is the prerequisite for SHAP relativities. The SHAP values are additive in log space, so exponentiating them gives multiplicative relativities directly analogous to `exp(beta)` from a GLM.

In [ ]:
features = ["area_code", "ncd_years", "has_convictions"]
X = df.select(features)

pool = catboost.Pool(
    data=X.to_pandas(),
    label=df["claim_count"].to_numpy(),
    weight=df["exposure"].to_numpy(),
)

model = catboost.CatBoostRegressor(
    loss_function="Poisson",
    iterations=300,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0,
)
model.fit(pool)
print("Training complete.")

## Extract SHAP relativities

`SHAPRelativities` wraps TreeSHAP from the `shap` library and adds the actuarial layer: exposure-weighted aggregation per factor level, CLT confidence intervals, and base-level normalisation so the output reads like a GLM factor table.

In [ ]:
from shap_relativities import SHAPRelativities

sr = SHAPRelativities(
    model=model,
    X=X,
    exposure=df["exposure"],
    categorical_features=features,
)
sr.fit()

rels = sr.extract_relativities(
    normalise_to="base_level",
    base_levels={"area_code": 0, "ncd_years": 0, "has_convictions": 0},
)

print(rels.select(["feature", "level", "relativity", "lower_ci", "upper_ci", "n_obs"]))

## Validate the extraction

Before using relativities in a rate filing, run `validate()`. The reconstruction check verifies that SHAP values sum back to the model's predictions to within floating-point tolerance. If this fails, the SHAP explainer was constructed incorrectly - almost always a mismatch between model objective and SHAP output type.

In [ ]:
checks = sr.validate()

for name, result in checks.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {name}: {result.message}")

## Compare to the true parameters

Because we constructed the data with known coefficients, we can measure relativity recovery directly. The key numbers:

- NCD=5 vs NCD=0: true discount is `exp(-0.12 * 5) = exp(-0.6) = 0.549`. SHAP typically recovers ~0.43-0.55 depending on data size and GBM depth.
- Conviction loading: true is `exp(0.45) = 1.57`. SHAP typically recovers 1.5-1.7.

On a log-linear DGP, the GLM has better relativity precision - it is correctly specified. The GBM advantage appears on portfolios with interaction effects. Even here, SHAP gives you level-specific factors with confidence intervals, which feature importance never can.

In [ ]:
import math

true_params = {
    ("ncd_years", "5"):       math.exp(-0.12 * 5),   # 0.549
    ("has_convictions", "1"): math.exp(0.45),         # 1.568
    ("area_code", "5"):       math.exp(0.65),         # 1.916
}

print(f"{'Feature':<20} {'Level':<8} {'True':>8} {'SHAP':>8} {'Error':>8}")
print("-" * 56)
for (feat, lvl), true_val in true_params.items():
    row = rels.filter((pl.col("feature") == feat) & (pl.col("level") == lvl))
    if len(row) > 0:
        shap_val = row["relativity"][0]
        err = (shap_val - true_val) / true_val * 100
        print(f"{feat:<20} {lvl:<8} {true_val:>8.3f} {shap_val:>8.3f} {err:>+7.1f}%")

## What you should see

The SHAP relativities are directionally correct - NCD=5 is cheaper than NCD=0, convictions load the price, higher area bands cost more. The exact values differ from the GLM by 5-25% depending on how well the GBM has converged on this small dataset.

The `lower_ci` and `upper_ci` columns are CLT confidence intervals based on data uncertainty (how precisely we have estimated each level's mean SHAP contribution). They do not capture model uncertainty from the GBM fitting process.

To export to a rating engine:

```python
rels.write_csv("relativities.csv")
# Columns: feature, level, relativity, lower_ci, upper_ci
# Maps directly to Radar/Emblem/Earnix CSV factor table import
```

**GitHub:** https://github.com/burning-cost/shap-relativities  
**PyPI:** https://pypi.org/project/shap-relativities/  
**Blog:** [Extracting Rating Relativities from GBMs with SHAP](https://burning-cost.github.io/2026/03/05/extracting-rating-relativities-from-gbms-with-shap/)